# Week 11 build · Reproduce one number

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-11.ipynb)

**~3 hours · CPU for Parts A–D and F · T4 for Part E · run top to bottom.**

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Type what you expect and how sure you are *before* running the experiment. The notebook refuses empty predictions. A rough guess is fine; the gap between guess and result is what you are here to collect (canvas steps 7–8: *predict before running*, *log the surprise*).

**Two targets.** `SMOKE = True` (config cell below) reproduces a number from *this course*: week 10's sound-probe accuracy, 71.1 %, as printed by the author's run. Everything in the reproduction workflow (checklist → run → tolerance → gap → report) is exercised on CPU in under five minutes with no download. `SMOKE = False` on a T4 adds Part E: the CLIP paper's Table 11 cell (ViT-B/32, zero-shot CIFAR-10, 91.3 %) with OpenCLIP's `openai` weights, about 800 MB of downloads.

| Part | Topic | Time |
|---|---|---|
| A | Three passes; the checklist as form fields; the tolerance | 25 min |
| B | The data, checked against the checklist | 20 min |
| C | The reproduction and its verdict | 25 min |
| D | Debugging order: seeds, steps, one deliberate mismatch | 35 min |
| E | T4: CLIP Table 11, 1 template vs 18, the wrong-weights swap | 45 min |
| F | The six-sentence report and the run-log row | 20 min |

In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

---
## Part A · Three passes, then the checklist (25 min)

**Words used in this part.** *checklist line*: one of five things a run must match: data, preprocessing, model, metric, the claimed number. *tolerance*: how far off still counts as the same number: two test-set standard errors, or the seed spread, whichever is larger. *claimed number*: the value in the table cell (or, for the rehearsal, the value the week-10 notebook printed on the author's machine).

Do the three passes on the laptop first (Lesson 1), then fill the two checklists. The rehearsal checklist is pre-filled from the week-10 notebook: check each line against the cells named, and edit anything that is wrong for *your* week-10 run.

In [ ]:
#@title Config: SMOKE = True reproduces this course's own number on CPU; False adds CLIP on a T4
SMOKE = True  #@param {type:"boolean"}
import json, math, os, platform, subprocess, sys, time, torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
RUN_CLIP = (not SMOKE) and torch.cuda.is_available()
cfg = dict(n_per_class=12, seeds=[0, 1, 2], probe_steps=200, seed=0)       # week 10's SMOKE config, verbatim
CLAIM = dict(number=71.1, per_seed=[68.3, 71.7, 73.3], sd=2.5, n_test=60,   # the "paper": the author's week-10 run
             source="week-10 notebook, cell 30, SMOKE=True, seeds 0-2",
             env="torch 2.14.0+cpu, numpy 2.5.3, Python 3.14, Windows, CPU")
if not SMOKE and not torch.cuda.is_available():
    print("SMOKE = False but no GPU: Part E would take about an hour on CPU. Runtime -> Change runtime type -> T4.")
print("SMOKE:", SMOKE, "| device:", device, "| Part E (CLIP) will run:", RUN_CLIP)
print("this run: torch", torch.__version__, "| numpy", np.__version__, "|", platform.python_version(), platform.system())

In [ ]:
#@title Part A · Checklist for the rehearsal target (week 10's number). One line each: what, and the cell it came from
data = "week10_data.npz: 20 classes x 12 per class = 240 images, render seed 0, SMOKE=True; 25% held out per probe seed (cells 3, 8, 28)"  #@param {type:"string"}
preprocessing = "image -> vOICe-style sound -> 256-number sound vector (cell 20); per-feature standardisation inside probe_accuracy (cell 28)"  #@param {type:"string"}
model = "one hidden layer of 64 with ReLU, Adam lr 1e-2, 200 steps on mini-batches of 64, torch seed = probe seed (cell 28)"  #@param {type:"string"}
metric = "held-out accuracy in %, one value per seed 0, 1, 2, then the mean (cell 30)"  #@param {type:"string"}
number = 71.1  #@param {type:"number"}
CHECKLIST = dict(data=data, preprocessing=preprocessing, model=model, metric=metric, number=number)
blanks = [k for k, v in CHECKLIST.items() if not str(v).strip()]
assert not blanks, f"checklist refused: empty lines {blanks}. An empty line is a detail you will discover after the run."
for k, v in CHECKLIST.items():
    print(f"{k:14s} {v}")

In [ ]:
#@title Part A · Checklist for the paper target (CLIP, Table 11). Fill from your three passes; page or file for each
clip_data = "CIFAR-10 test split, 10,000 images, 10 classes (Table 9, p. 39)"  #@param {type:"string"}
clip_preprocessing = "preprocess shipped with the weights: resize 224, centre crop, CLIP mean/std (clip.py); 18 templates from openai/CLIP data/prompts.md averaged in embedding space (section 3.1.4)"  #@param {type:"string"}
clip_model = "ViT-B/32 with OpenAI's released weights: open_clip pretrained='openai', NOT laion2b (Table 11 row)"  #@param {type:"string"}
clip_metric = "top-1 accuracy over the 10,000 test images (Table 9)"  #@param {type:"string"}
clip_number = 91.3  #@param {type:"number"}
CLIP_CHECKLIST = dict(data=clip_data, preprocessing=clip_preprocessing, model=clip_model, metric=clip_metric, number=clip_number)
blanks = [k for k, v in CLIP_CHECKLIST.items() if not str(v).strip()]
assert not blanks, f"checklist refused: empty lines {blanks}"
for k, v in CLIP_CHECKLIST.items():
    print(f"{k:14s} {v}")

In [ ]:
#@title 🔮 Predict A1: Tolerance band for the rehearsal target (71.1% on 60 held-out images, seed sd 2.5): under 3 points, 3-10, or over 10? And for the CLIP cell (91.3% on 10,000 images)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A1", prediction, confidence)

In [ ]:
def tolerance(p, n, sd=0.0):
    """Band in points: 2 x max(test-set standard error, seed spread). p in %, n = images the number was measured on."""
    se = math.sqrt(p / 100 * (1 - p / 100) / n) * 100
    return 2 * max(se, sd), se

band, se = tolerance(CLAIM["number"], CLAIM["n_test"], CLAIM["sd"])
print(f"rehearsal target: {CLAIM['number']}% on {CLAIM['n_test']} held-out images -> SE {se:.1f} pts, seed sd {CLAIM['sd']} -> band +-{band:.1f}")
band_clip, se_clip = tolerance(CLIP_CHECKLIST["number"], 10000)
print(f"CLIP cell: {CLIP_CHECKLIST['number']}% on 10,000 images, no seed -> SE {se_clip:.2f} pts -> band +-{band_clip:.1f}")
print(f"\nfor scale: OpenCLIP's own table reports 89.8 for the same weights, {abs(89.8 - 91.3) / se_clip:.1f} standard errors from the paper's 91.3")

In [ ]:
#@title ✍️ A1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A1", what_happened, surprise)

Surprise? The rehearsal target has a band of about ±12 points: 60 held-out images and three seeds cannot pin a number down. Almost any honest rerun will "reproduce" it, and that is the first finding of the week: the claimed 71.1 was never precise. The CLIP cell is the opposite case: 10,000 images make a band of ±0.6, so a run at 89.8 is a real gap that needs an explanation, not noise.

---
## Part B · The data, checked against the checklist (20 min)

**Words used in this part.** *entry point*: the cell order of a notebook, or the script a README names. *config file*: week 10's `cfg` cell: `n_per_class`, `seeds`, `probe_steps`, `seed`. *data flow*: render → sonify → sound vector → standardise → probe → held-out accuracy. *checklist mismatch*: a fact about the data file that contradicts a checklist line, found before the run instead of after.

Week 10's notebook saved `week10_data.npz`. If you uploaded it to this runtime (Files panel), it is used as-is and checked; if not, the four cells below rebuild it with week 10's code at SMOKE size. Either way the file's own fields (`smoke`, `seed`, `option`, size) are compared with the checklist.

In [ ]:
yy, xx = np.mgrid[0:16, 0:16].astype(float)
def _m(dx, dy, r):
    return dict(ax=np.abs(dx), ay=np.abs(dy), d2=dx**2 + dy**2, r=r, dx=dx, dy=dy)
SHAPES = {
 "circle":     lambda m: m["d2"] <= m["r"]**2,
 "ring":       lambda m: (m["d2"] <= m["r"]**2) & (m["d2"] >= (0.6 * m["r"])**2),
 "square":     lambda m: np.maximum(m["ax"], m["ay"]) <= m["r"],
 "hollow sq":  lambda m: (np.maximum(m["ax"], m["ay"]) <= m["r"]) & (np.maximum(m["ax"], m["ay"]) > 0.6 * m["r"]),
 "diamond":    lambda m: m["ax"] + m["ay"] <= m["r"],
 "tri up":     lambda m: (m["ay"] <= m["r"]) & (m["ax"] <= (m["dy"] + m["r"]) * 0.6),
 "tri down":   lambda m: (m["ay"] <= m["r"]) & (m["ax"] <= (m["r"] - m["dy"]) * 0.6),
 "plus":       lambda m: ((m["ax"] <= 1.2) & (m["ay"] <= m["r"])) | ((m["ay"] <= 1.2) & (m["ax"] <= m["r"])),
 "X":          lambda m: (np.abs(m["ax"] - m["ay"]) <= 1.2) & (np.maximum(m["ax"], m["ay"]) <= m["r"]),
 "h bar":      lambda m: (m["ay"] <= 1.2) & (m["ax"] <= m["r"]),
}

In [ ]:
SHAPES.update({
 "v bar":      lambda m: (m["ax"] <= 1.2) & (m["ay"] <= m["r"]),
 "rising":     lambda m: (np.abs(m["dx"] + m["dy"]) <= 1.4) & (np.maximum(m["ax"], m["ay"]) <= m["r"]),
 "falling":    lambda m: (np.abs(m["dx"] - m["dy"]) <= 1.4) & (np.maximum(m["ax"], m["ay"]) <= m["r"]),
 "L":          lambda m: ((np.abs(m["dx"] + m["r"]) <= 1.2) & (m["ay"] <= m["r"])) | ((np.abs(m["dy"] - m["r"]) <= 1.2) & (m["ax"] <= m["r"])),
 "T":          lambda m: ((np.abs(m["dy"] + m["r"]) <= 1.2) & (m["ax"] <= m["r"])) | ((m["ax"] <= 1.2) & (m["ay"] <= m["r"])),
 "U":          lambda m: ((np.abs(m["ax"] - m["r"]) <= 1.2) & (m["ay"] <= m["r"])) | ((np.abs(m["dy"] - m["r"]) <= 1.2) & (m["ax"] <= m["r"])),
 "dot":        lambda m: m["d2"] <= (0.45 * m["r"])**2,
 "half disc":  lambda m: (m["d2"] <= m["r"]**2) & (m["dy"] <= 0),
 "checker":    lambda m: (np.maximum(m["ax"], m["ay"]) <= m["r"]) & ((m["dx"] > 0) ^ (m["dy"] > 0)),
 "corners":    lambda m: (np.maximum(m["ax"], m["ay"]) <= m["r"]) & (m["ax"] > 0.5 * m["r"]) & (m["ay"] > 0.5 * m["r"]),
})
CLASSES = list(SHAPES)
print(len(CLASSES), "classes (week 10's renderer, copied verbatim)")

In [ ]:
def render(name, rng, jitter=True):
    """One 16x16 image in [0, 1]: shape = 1, background = 0, plus faint sensor noise (week 10, cell 6)."""
    cx = cy = 7.5; r = 5.0
    if jitter:
        cx, cy = 7.5 + rng.uniform(-2, 2), 7.5 + rng.uniform(-2, 2); r = rng.uniform(4.0, 6.0)
    img = SHAPES[name](_m(xx - cx, yy - cy, r)).astype(float)
    return np.clip(img + 0.03 * rng.standard_normal((16, 16)), 0, 1)

def make_dataset(n_per_class, seed):
    rng = np.random.default_rng(seed)
    X = np.stack([render(c, rng) for c in CLASSES for _ in range(n_per_class)]).astype(np.float32)
    y = np.repeat(np.arange(len(CLASSES)), n_per_class)
    perm = rng.permutation(len(y))
    return X[perm], y[perm]

In [ ]:
sr, n_rows, col_dur = 8000, 16, 0.06                                       # week 10, cell 15: the sonifier
t = np.arange(int(sr * col_dur)) / sr
freqs = np.logspace(np.log10(2800), np.log10(350), n_rows)
W = np.sin(2 * np.pi * freqs[None, :] * t[:, None]).astype(np.float32)
NFFT = 512; fft_hz = np.fft.rfftfreq(NFFT, 1 / sr)
edges = np.logspace(np.log10(300), np.log10(3300), n_rows + 1)
BANDS = [(fft_hz >= lo) & (fft_hz < hi) for lo, hi in zip(edges[:-1], edges[1:])]
window = np.hanning(len(t)).astype(np.float32)

def sonify(img):
    return (W @ img).T.reshape(-1)

def sound_map(audio):
    power = np.abs(np.fft.rfft(audio.reshape(16, -1) * window, n=NFFT)) ** 2
    e = np.sqrt(np.stack([power[:, b].sum(1) for b in BANDS], axis=1))
    return (e / (e.max() + 1e-8)).T[::-1]
print("sonifier ready: one image ->", sonify(np.zeros((16, 16), dtype=np.float32)).shape, "samples -> 16x16 sound map")

In [ ]:
#@title 🔮 Predict B1: Is week10_data.npz already in this runtime? If it has to be regenerated, will the images match week 10's bit for bit (same seed 0)? And will the file pass every checklist line, or which one fails?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
PATH = "week10_data.npz"
if not os.path.exists(PATH):
    print("week10_data.npz not found: regenerating with week 10's code at SMOKE size (n_per_class = 12, seed 0)")
    imgs, labs = make_dataset(cfg["n_per_class"], cfg["seed"])
    svecs = np.stack([sound_map(sonify(im)).ravel() for im in imgs]).astype(np.float32)
    np.savez_compressed(PATH, images=imgs, sound=svecs, labels=labs, classes=np.array(CLASSES),
                        seed=cfg["seed"], option="contrastive", smoke=True, regenerated=True)
d = np.load(PATH)
images, sound_vecs, labels, CLASSES = d["images"], d["sound"], d["labels"], [str(c) for c in d["classes"]]
N, n_per = len(labels), len(labels) // len(CLASSES)
regen_imgs, _ = make_dataset(cfg["n_per_class"], cfg["seed"])                  # what the checklist says the data is
same_bits = images.shape == regen_imgs.shape and np.array_equal(images, regen_imgs)
mismatches = []
if not bool(d["smoke"]): mismatches.append(f"file made with SMOKE=False ({n_per} per class); the claimed number used 12 per class")
if int(d["seed"]) != cfg["seed"]: mismatches.append(f"render seed {int(d['seed'])} != {cfg['seed']}")
if n_per != cfg["n_per_class"]: mismatches.append(f"{n_per} per class != {cfg['n_per_class']}")
if not same_bits: mismatches.append("images differ from a fresh render with seed 0 (different numpy? different SMOKE size?)")
print(f"{N} images ({n_per} per class), {len(CLASSES)} classes, seed {int(d['seed'])}, smoke={bool(d['smoke'])}, "
      f"option={str(d['option'])}, regenerated here: {bool(d['regenerated']) if 'regenerated' in d else False}")
print("bit-identical to a fresh seed-0 render:", same_bits)
print("checklist mismatches:", mismatches or "none: the data line holds")

In [ ]:
#@title ✍️ B1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B1", what_happened, surprise)

Surprise? A fresh render with seed 0 matches the saved images bit for bit: NumPy's `default_rng` gives the same stream on every platform, so the *data* line of the checklist is the easy one. If you uploaded a file made with `SMOKE = False`, the cell flagged it: 60 images per class is a different dataset from the one the claimed number was measured on, and the run would have compared apples with oranges without saying so. That flag, before the run, is what Part B is for.

---
## Part C · The reproduction and its verdict (25 min)

**Words used in this part.** *reproduction*: the same number from the same recipe, judged against the tolerance. *held-out accuracy*: the probe's accuracy on the 25 % of images its seed kept out of training. *verdict*: inside or outside the band, by how much.

The probe is week 10's `probe_accuracy`, copied verbatim, with one added switch (`standardise`) that Part D will flip. Same seeds, same steps, same data: the only thing that can differ is your machine.

In [ ]:
def probe_accuracy(X, y, seed, steps=None, hidden=64, lr=1e-2, standardise=True):
    """Week 10, cell 28: train a one-hidden-layer probe on 75% with this seed; return held-out accuracy in %."""
    steps = steps or cfg["probe_steps"]
    g = torch.Generator().manual_seed(seed); torch.manual_seed(seed)
    Xt, yt = torch.tensor(X.reshape(len(X), -1), dtype=torch.float32), torch.tensor(y)
    if standardise:
        Xt = (Xt - Xt.mean(0)) / (Xt.std(0) + 1e-6)
    perm = torch.randperm(len(yt), generator=g); n_tr = int(0.75 * len(yt))
    tr, te = perm[:n_tr], perm[n_tr:]
    net = nn.Sequential(nn.Linear(Xt.shape[1], hidden), nn.ReLU(), nn.Linear(hidden, len(CLASSES)))
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for _ in range(steps):
        idx = tr[torch.randperm(n_tr, generator=g)[:64]]
        loss = F.cross_entropy(net(Xt[idx]), yt[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return float((net(Xt[te]).argmax(1) == yt[te]).float().mean()) * 100

print("probe ready | seeds:", cfg["seeds"], "| steps:", cfg["probe_steps"], "| held-out images per seed:", N - int(0.75 * N))

In [ ]:
#@title 🔮 Predict C1: Same seeds 0, 1, 2, same code, your torch version: will the mean land inside the +-band around 71.1? And will the three per-seed numbers match the author's [68.3, 71.7, 73.3] digit for digit?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
t0 = time.time()
accs = [probe_accuracy(sound_vecs, labels, s) for s in cfg["seeds"]]
mean, sd = float(np.mean(accs)), float(np.std(accs, ddof=1))
gap = mean - CLAIM["number"]
verdict = "WITHIN tolerance" if abs(gap) <= band else "OUTSIDE tolerance"
for s, a, b in zip(cfg["seeds"], accs, CLAIM["per_seed"]):
    print(f"seed {s}: yours {a:5.1f}%   author {b:5.1f}%   {'identical' if abs(a - b) < 0.05 else f'differs by {a - b:+.1f}'}")
print(f"\nyours: mean {mean:.1f}  sd {sd:.1f}   |   claimed {CLAIM['number']} +- {band:.1f}   |   gap {gap:+.1f}  ->  {verdict}")
print(f"this run : torch {torch.__version__}, numpy {np.__version__}, {platform.system()} {device}")
print(f"claimed  : {CLAIM['env']}")
RESULT = dict(mean=mean, sd=sd, gap=gap, verdict=verdict, per_seed=[round(a, 1) for a in accs], seconds=round(time.time() - t0, 1))

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

Surprise? Two different questions were answered. *Within tolerance* is almost guaranteed by a ±12 band. *Digit for digit* is not: the same seed gives the same stream of random numbers only for the same PyTorch version on the same kind of hardware; a different version, a GPU, or a different BLAS can change the initial weights or the order of floating-point additions, and 200 Adam steps amplify the difference. "Same seed" pins a run on one machine, not across machines. That is why the criterion is a tolerance, and why the report's setup sentence names the library versions.

---
## Part D · Debugging order (35 min)

**Words used in this part.** *debugging order*: metric, data, preprocessing, model, seeds, versions, hardware; cheapest and likeliest first. *seed spread*: the sd across runs that differ only in the seed. *measured move*: how much the number changed when one thing changed; the only kind of evidence the explanation sentence accepts.

Steps 1, 2 and 6 were settled in Parts A–C (same metric, same data bit for bit, versions printed). Three runs remain: step 5 (more seeds), a config change the paper-equivalent hid (`probe_steps` in the full run), and one deliberate preprocessing mismatch to see what step 3 looks like when it is the culprit. Every move goes into `MOVES` for the report.

In [ ]:
#@title 🔮 Predict D1: Seeds 0-9 instead of 0-2: will the ten-seed mean be higher, lower or about the same as 71.1, and how wide the spread (sd under 3, 3-6, over 6)? Does the claimed 71.1 sit inside mean +- 2 sd?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", prediction, confidence)

In [ ]:
MOVES = {}
accs10 = [probe_accuracy(sound_vecs, labels, s) for s in range(10)]
mean10, sd10 = float(np.mean(accs10)), float(np.std(accs10, ddof=1))
MOVES["seeds 0-9 instead of 0-2"] = mean10 - RESULT["mean"]
print("per seed:", " ".join(f"{a:.1f}" for a in accs10))
print(f"ten seeds: mean {mean10:.1f}  sd {sd10:.1f}   (three seeds: mean {RESULT['mean']:.1f}  sd {RESULT['sd']:.1f})")
inside = abs(CLAIM["number"] - mean10) <= 2 * sd10
print(f"claimed {CLAIM['number']} sits {'inside' if inside else 'outside'} the ten-seed mean +- 2 sd  [{mean10 - 2 * sd10:.1f}, {mean10 + 2 * sd10:.1f}]")
print(f"step 5 move: {MOVES['seeds 0-9 instead of 0-2']:+.1f} points; seeds 0, 1, 2 rank "
      f"{sorted(range(10), key=lambda i: -accs10[i]).index(0) + 1}, {sorted(range(10), key=lambda i: -accs10[i]).index(1) + 1}, "
      f"{sorted(range(10), key=lambda i: -accs10[i]).index(2) + 1} of 10")
plt.plot(range(10), accs10, "o"); plt.axhline(CLAIM["number"], ls="--", c="grey", label="claimed"); plt.axhline(mean10, c="C1", label="10-seed mean")
plt.xlabel("seed"); plt.ylabel("held-out accuracy %"); plt.legend(); plt.show()

In [ ]:
#@title ✍️ D1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict D2: 500 steps instead of 200 (the config the full week-10 run uses): does the three-seed mean move by more than the seed spread? Up or down?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D2", prediction, confidence)

In [ ]:
accs500 = [probe_accuracy(sound_vecs, labels, s, steps=500) for s in cfg["seeds"]]
MOVES["500 steps instead of 200"] = float(np.mean(accs500)) - RESULT["mean"]
print("500 steps, seeds 0-2:", " ".join(f"{a:.1f}" for a in accs500), f"-> mean {np.mean(accs500):.1f}")
print(f"move: {MOVES['500 steps instead of 200']:+.1f} points; seed spread at 200 steps was {RESULT['sd']:.1f}, at ten seeds {sd10:.1f}")
print("verdict on this config line:", "hidden setting matters" if abs(MOVES["500 steps instead of 200"]) > sd10 else "inside the seed spread: a config detail the number does not depend on")

In [ ]:
#@title ✍️ D2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D2", what_happened, surprise)

In [ ]:
#@title 🔮 Predict D3: Deliberate preprocessing mismatch: skip the per-feature standardisation inside the probe. Does the mean move up or down, and by how much: under 2, 2-10, or over 10 points? Will it look like a seed effect or unmistakably like preprocessing?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D3", prediction, confidence)

In [ ]:
accs_raw = [probe_accuracy(sound_vecs, labels, s, standardise=False) for s in cfg["seeds"]]
MOVES["no standardisation (deliberate mismatch)"] = float(np.mean(accs_raw)) - RESULT["mean"]
print("no standardisation, seeds 0-2:", " ".join(f"{a:.1f}" for a in accs_raw), f"-> mean {np.mean(accs_raw):.1f}")
print(f"move: {MOVES['no standardisation (deliberate mismatch)']:+.1f} points   (band +-{band:.1f}, ten-seed sd {sd10:.1f})")
print("\nmoves so far, largest first:")
for k, v in sorted(MOVES.items(), key=lambda kv: -abs(kv[1])):
    print(f"  {v:+6.1f}  {k}")

In [ ]:
#@title ✍️ D3: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D3", what_happened, surprise)

**Reading the moves.** Rank them by size against the band. A move inside the seed spread is not an explanation, whatever its sign; a move larger than the band is a candidate, and the one that also matches the gap's size and direction is the explanation. The deliberate mismatch shows what a preprocessing change looks like from the outside: every seed moves the same way, which randomness does not do, and the direction can surprise you (skipping standardisation may *help* this probe: the sound vectors are already scaled to [0, 1], and standardising amplifies the near-silent, noisy features). A change that helps is still a mismatch: the target is the paper's recipe, not the best number. In your own reproductions you will not know which step is the culprit; the order tells you which to try first.

---
## Part E · T4: the CLIP cell (45 min; skipped when SMOKE = True or there is no GPU)

**Words used in this part.** *zero-shot classification*: embed each class as a sentence, embed the image, pick the closest sentence. *prompt template*: the sentence the class name is dropped into; the paper averages 18 of them for CIFAR-10. *weights id*: the `pretrained` string; `openai` are the paper's weights, `laion2b_s34b_b79k` are a different model with the same architecture.

Set `SMOKE = False` on a T4 and rerun from the config cell. Downloads: OpenCLIP weights (about 600 MB) and CIFAR-10 (170 MB). The full 10,000-image test set is scored, because the checklist says 10,000: scoring 2,000 would widen the band to ±1.3 and quietly change the target.

In [ ]:
if RUN_CLIP:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch", "torchvision"], check=True)
    import open_clip, torchvision
    print("downloading ViT-B-32 'openai' weights (~600 MB) and CIFAR-10 test (~170 MB) ...")
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")   # the paper's weights
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    model = model.to(device).eval()
    cifar = torchvision.datasets.CIFAR10(root="data", train=False, download=True, transform=preprocess)
    loader = torch.utils.data.DataLoader(cifar, batch_size=256, num_workers=2)
    print(f"ready: {len(cifar)} test images (checklist says 10,000) | classes: {cifar.classes}")
    print("step 2-4 prints:", cifar[0][0].shape, f"{cifar[0][0].min():.2f}..{cifar[0][0].max():.2f}", "| eval mode:", not model.training)
else:
    print("Part E skipped (SMOKE = True, or no GPU).")

In [ ]:
TEMPLATES_18 = ["a photo of a {}.", "a blurry photo of a {}.", "a black and white photo of a {}.", "a low contrast photo of a {}.",
                "a high contrast photo of a {}.", "a bad photo of a {}.", "a good photo of a {}.", "a photo of a small {}.",
                "a photo of a big {}.", "a photo of the {}.", "a blurry photo of the {}.", "a black and white photo of the {}.",
                "a low contrast photo of the {}.", "a high contrast photo of the {}.", "a bad photo of the {}.",
                "a good photo of the {}.", "a photo of the small {}.", "a photo of the big {}."]   # openai/CLIP data/prompts.md, CIFAR10

def zero_shot_accuracy(model, tokenizer, templates, classes, loader):
    """Text features per class = mean over templates (in embedding space, as section 3.1.4 says); top-1 over the loader."""
    with torch.no_grad():
        txt = torch.stack([F.normalize(model.encode_text(tokenizer([t.format(c) for t in templates]).to(device)), dim=-1).mean(0)
                           for c in classes])
        txt = F.normalize(txt, dim=-1)
        hits = n = 0
        for x, y in loader:
            img = F.normalize(model.encode_image(x.to(device)), dim=-1)
            hits += int((img @ txt.T).argmax(1).cpu().eq(y).sum()); n += len(y)
    return 100 * hits / n, n
print("zero-shot scorer ready |", len(TEMPLATES_18), "templates")

In [ ]:
#@title 🔮 Predict E1: OpenCLIP 'openai' ViT-B/32 on the 10,000 CIFAR-10 test images: with the single template 'a photo of a {}.' and with all 18 averaged. Which is higher, by how much, and does either land within +-0.6 of 91.3?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E1", prediction, confidence)

In [ ]:
if RUN_CLIP:
    t0 = time.time()
    acc1, n1 = zero_shot_accuracy(model, tokenizer, TEMPLATES_18[:1], cifar.classes, loader)
    acc18, n18 = zero_shot_accuracy(model, tokenizer, TEMPLATES_18, cifar.classes, loader)
    gap_clip = acc18 - CLIP_CHECKLIST["number"]
    verdict_clip = "WITHIN tolerance" if abs(gap_clip) <= band_clip else "OUTSIDE tolerance"
    MOVES["1 template instead of 18 (CLIP)"] = acc1 - acc18
    print(f"1 template : {acc1:.2f}% on {n1} images")
    print(f"18 templates: {acc18:.2f}% on {n18} images   |   claimed 91.3 +- {band_clip:.1f}   |   gap {gap_clip:+.2f}  ->  {verdict_clip}")
    print(f"OpenCLIP's own table for these weights: 89.8   |   {time.time() - t0:.0f}s")
    CLIP_RESULT = dict(acc1=acc1, acc18=acc18, gap=gap_clip, verdict=verdict_clip, n=n18, open_clip=open_clip.__version__)
else:
    print("Part E skipped (SMOKE = True, or no GPU).")

In [ ]:
#@title ✍️ E1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict E2: Swap to the laion2b_s34b_b79k weights (week 9's choice), same 18 templates: above or below 91.3, and by how much? Is that a reproduction of Table 11?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E2", prediction, confidence)

In [ ]:
if RUN_CLIP:
    model_l, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
    model_l = model_l.to(device).eval()
    acc_l, _ = zero_shot_accuracy(model_l, tokenizer, TEMPLATES_18, cifar.classes, loader)
    MOVES["laion2b weights instead of openai (CLIP)"] = acc_l - CLIP_RESULT["acc18"]
    print(f"laion2b weights, 18 templates: {acc_l:.2f}%   (openai weights: {CLIP_RESULT['acc18']:.2f}%, paper: 91.3)")
    print(f"move: {MOVES['laion2b weights instead of openai (CLIP)']:+.2f} points. A better number is still not a reproduction: different weights, different model line.")
    del model_l; torch.cuda.empty_cache()
else:
    print("Part E skipped (SMOKE = True, or no GPU).")

In [ ]:
#@title ✍️ E2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E2", what_happened, surprise)

Surprise? Whatever the exact numbers, three things are now measured instead of guessed: how much the 18-template ensemble is worth on CIFAR-10, how far the `openai` weights land from the paper's cell, and how far the wrong weights land in the other direction. If the 18-template run sits outside ±0.6, the residue is real and the next suspects are step 3 (the resize method inside `preprocess`) and step 7 (fp16 versus fp32); both cost one run.

---
## Part F · The report and the run-log row (20 min)

**Words used in this part.** *reproduction report*: six sentences: target, setup, result, verdict, explanation, residue. *residue*: the part of the gap no measured move accounts for, with the next check and its cost. *run log*: one JSON row per run: time, code version, config, seed, versions, result.

The cell below writes the paragraph from the numbers this notebook recorded (the rehearsal by default, the CLIP cell when Part E ran). Edit the two free-text fields: the explanation may only quote moves from `MOVES`; the residue must name the next check and its cost.

In [ ]:
#@title Part F · The two sentences the notebook cannot write for you
explanation = ""  #@param {type:"string"}
residue = ""  #@param {type:"string"}
use_clip = RUN_CLIP and "CLIP_RESULT" in globals()
if not explanation.strip():                                  # default: the measured moves, largest first
    explanation = "; ".join(f"{k} moves it {v:+.1f}" for k, v in sorted(MOVES.items(), key=lambda kv: -abs(kv[1]))) + " (all measured)"
if not residue.strip():
    left = (CLIP_RESULT["gap"] if use_clip else RESULT["gap"])
    residue = (f"{abs(left):.1f} points unexplained; next check: resize method inside preprocess, one run, 3 min" if use_clip
               else f"{abs(left):.1f} points, inside the band; not pursued. Next check if it mattered: 10 seeds x 500 steps, 2 min")
print("explanation:", explanation); print("residue    :", residue)

In [ ]:
try:
    commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip() or "n/a"
except Exception:
    commit = "n/a (not a git checkout)"
if use_clip:
    S = [f"Target: Radford et al. 2021, Table 11, ViT-B/32 zero-shot CIFAR-10, {CLIP_CHECKLIST['number']}.",
         f"Setup: open_clip {CLIP_RESULT['open_clip']}, pretrained=openai, {CLIP_RESULT['n']} test images, shipped preprocess, 18 templates averaged, "
         f"{torch.cuda.get_device_name(0)} fp32, torch {torch.__version__}, commit {commit}, no seed (deterministic).",
         f"Result: {CLIP_RESULT['acc18']:.2f}, tolerance +-{band_clip:.1f} (test-set SE {se_clip:.2f}).",
         f"Verdict: {CLIP_RESULT['verdict']} (gap {CLIP_RESULT['gap']:+.2f})."]
else:
    S = [f"Target: {CLAIM['source']}, sound-probe held-out accuracy, {CLAIM['number']}.",
         f"Setup: week10_data.npz ({N} images, seed {cfg['seed']}, smoke={bool(d['smoke'])}), standardise + 75/25 split, 64-unit probe, "
         f"{cfg['probe_steps']} Adam steps, seeds {cfg['seeds']}, {device}, torch {torch.__version__}, numpy {np.__version__}, commit {commit}.",
         f"Result: {RESULT['mean']:.1f} (per seed {RESULT['per_seed']}), tolerance +-{band:.1f} (test-set SE {se:.1f}, seed sd {CLAIM['sd']}).",
         f"Verdict: {RESULT['verdict']} (gap {RESULT['gap']:+.1f})."]
S += [f"Explanation: {explanation}.", f"Residue: {residue}."]
report = " ".join(S)
open("week11_report.md", "w", encoding="utf-8").write("# Week 11 reproduction report\n\n" + report + "\n")
print(report); print("\nsaved week11_report.md (download it from the Files panel and commit it next to week10_plan.md)")

In [ ]:
row = dict(time=time.strftime("%Y-%m-%d %H:%M"), lesson="week-11/build", target="clip-table11" if use_clip else "week10-sound-probe",
           commit=commit, config=cfg, seeds=cfg["seeds"], torch=torch.__version__, numpy=np.__version__, device=device,
           result=(CLIP_RESULT["acc18"] if use_clip else RESULT["mean"]), verdict=(CLIP_RESULT["verdict"] if use_clip else RESULT["verdict"]),
           moves={k: round(v, 2) for k, v in MOVES.items()})
with open("runs.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(row) + "\n")
print("run-log row appended to runs.jsonl:"); print(json.dumps(row, indent=1)[:900])

---
## Wrap up

Run the cell below, copy the table into your insight ledger next to `core-tooling` (and the verdict and residue sentences next to `lin-clip`), and mark the biggest surprise. Then on the laptop:

```
python track.py done week-11/build --rating <1-5> --minutes 180 --note "biggest surprise"
python gen_week.py 12
```

In [ ]:
reveal()